## Вычисление минимального детектируемого эффекта: симуляция Монте-Карло

**Условие кейса:** К нам пришли наши коллеги из ML-отдела и рассказали, что планируется выкатывать новый алгоритм, рекомендующий пользователям интересные посты. После обсуждений того, как он это делает, вы пришли к следующему пониманию:

1. Алгоритм добавляет пользователям 1-2 просмотра
2. Вероятность того, что он сработает, составляет 90%
3. Если у пользователя меньше 50 просмотров, то алгоритм не сработает

Предполагается, что  увеличение числа просмотров приведёт и к увеличению лайков на пользователя.

**Вопрос: сможем ли мы обнаружить различия в среднем количестве лайков на пользователя?**

Проведу симуляцию Монте-Карло.

### Исходные данные теста

In [1]:
import pandas as pd
import pandahouse as ph

In [2]:
#параметры соединения - нужны, чтобы подключиться к нужной схеме данных
connection = {'host': 'http://clickhouse.lab.karpov.courses:8123',
'database':'simulator_20260620',
'user':'student',
'password':'dpo_python_2020'
}

Вводные данные симуляции :
- Симулируем просмотры и CTR из реальных данных. Реальные данные принимаются за генеральную совокупность, а выборки семлирую.
- Даты проведения эксперимента: 22.05.2026 - 28.05.2026 (1 неделя)
- Количество симуляций задайте не меньше 20000
- Эффект алгоритма на просмотры: group_B_views + ((1 + rng.binomial(n=1, p=0.5, size=размер_выборки)) * rng.binomial(n=1, p=0.9, size=размер_выборки) * (group_B_views >= 50))
- Лайки сравниваем t-тестом с поправкой Уэлча на неравные дисперсии
- Уровень значимости - 0.05
- Делим пользователей на группы 50/50

**Вычисляем мощность теста**

In [3]:
q = """
SELECT views, count() as users
FROM (SELECT 
    user_id,
    sum(action = 'view') as views
FROM simulator_20260620.feed_actions 
WHERE toDate(time) between '2026-05-22' and '2026-05-28'
GROUP BY user_id
)
GROUP BY views
ORDER BY views
"""

views_distribution = ph.read_clickhouse(q, connection=connection)

In [4]:
views_distribution.head() 

# смотрим распределение пользователей по количеству просмотров

,views,users
0,1,4
1,2,1
2,3,4
3,4,5
4,5,18


Вычисляю долю просмотров от общего числа пользователей по дням, чтобы получить распределение вероятностей на реальных данных.

In [5]:
views_distribution['pr'] = views_distribution['users']/views_distribution.users.sum()

In [6]:
views_distribution.sort_values('pr', ascending = False)

,views,users,pr
15,16,545,0.012977
14,15,537,0.012787
13,14,500,0.011906
34,35,485,0.011548
29,30,469,0.011167
...,...,...,...
280,287,1,0.000024
278,285,1,0.000024
276,280,1,0.000024
1,2,1,0.000024


In [7]:
q = """
select toDate(time) as dt,
    action,
    user_id,
    sum(action = 'like')/sum(action = 'view') as ctr
from simulator_20260620.feed_actions
where toDate(time) between '2026-05-22' and '2026-05-28'
group by dt, user_id, action
"""

draft = ph.read_clickhouse(q, connection=connection)

In [8]:
draft.head()

,dt,action,user_id,ctr
0,2026-05-22,view,114432,0.0
1,2026-05-27,like,131563,inf
2,2026-05-27,view,107122,0.0
3,2026-05-28,view,8320,0.0
4,2026-05-23,like,17418,inf


In [9]:
q = """
select 
   floor(ctr, 2) as ctr, count() as users
from (select toDate(time) as dt,
    user_id,
    sum(action = 'like')/sum(action = 'view') as ctr
from simulator_20260620.feed_actions
where toDate(time) between '2026-05-22' and '2026-05-28'
group by dt, user_id
)
group by ctr
"""


ctr_distribution = ph.read_clickhouse(q, connection=connection)

In [10]:
ctr_distribution.head() 

,ctr,users
0,0.00,1443
1,0.65,4
2,0.71,5
3,0.49,4
4,0.54,72


На реальных данных вычисляю долю пользователей с каждым значением ctr.

In [11]:
ctr_distribution['pr'] = ctr_distribution['users']/ctr_distribution.users.sum()

In [12]:
ctr_distribution.sort_values('ctr', ascending = False)

,ctr,users,pr
51,1.00,1,0.000012
39,0.88,1,0.000012
26,0.85,3,0.000035
73,0.83,1,0.000012
12,0.81,2,0.000023
...,...,...,...
16,0.05,727,0.008541
32,0.04,312,0.003665
23,0.03,142,0.001668
69,0.02,48,0.000564


In [13]:
ctr = ctr_distribution['ctr'].to_numpy()
ctr_prob = ctr_distribution['pr'].to_numpy()

In [14]:
ctr

array([0.  , 0.65, 0.71, 0.49, 0.54, 0.23, 0.18, 0.72, 0.28, 0.07, 0.5 ,
       0.51, 0.81, 0.33, 0.38, 0.7 , 0.05, 0.2 , 0.37, 0.73, 0.8 , 0.48,
       0.12, 0.03, 0.45, 0.62, 0.85, 0.69, 0.68, 0.17, 0.58, 0.59, 0.04,
       0.16, 0.42, 0.64, 0.26, 0.3 , 0.22, 0.88, 0.47, 0.41, 0.27, 0.09,
       0.36, 0.46, 0.55, 0.56, 0.14, 0.61, 0.25, 1.  , 0.39, 0.43, 0.66,
       0.53, 0.19, 0.35, 0.75, 0.76, 0.1 , 0.4 , 0.57, 0.24, 0.06, 0.31,
       0.63, 0.34, 0.29, 0.02, 0.08, 0.32, 0.21, 0.83, 0.13, 0.52, 0.15,
       0.44, 0.6 , 0.11])

В условии сказано, что нужны все данные. И что пользователи будут разделены 50/50 на группы.

In [15]:
N = int(views_distribution['users'].sum()/2)

N 
# количество пользователей в каждой из двух групп

20998

In [16]:
import numpy as np

rng = np.random.default_rng()

In [17]:
from scipy import stats

Дизайн эксперимента:
- Семплирую просмотры для обеих групп из реальных данных: group_A_views и group_B_views
- Беру группу B за экспериментальную (на нее распространяется изменение)
- Рассчитываю ожидаемый эффект на просмотры (group_B)
- Семплирую ctr для обеих групп: group_A_CTR и group_B_CTR
- Создаю биномиальное распределение для просмотров пользователей группы А и просмотров пользователей группы B с эффектом. CTR берется для оценки вероятности клика.
- Провожу t-тест с учетом неравенства дисперсий

In [18]:
from tqdm import tqdm

In [19]:
p_values = []
for _ in tqdm(range(20000)): 
    group_A_views = rng.choice(views_distribution['views'], size= N, 
                           replace=True, p=views_distribution['pr'])
    group_B_views = rng.choice(views_distribution['views'], size=N, 
                           replace=True, p=views_distribution['pr'])
    group_B = group_B_views + ((1 + rng.binomial(n=1, p=0.5, size=N)) * rng.binomial(n=1, p=0.9, size=N) * (group_B_views >= 50))
    group_A_CTR=rng.choice(ctr, size=N, replace=True, p=ctr_prob)
    group_B_CTR=rng.choice(ctr, size=N, replace=True, p=ctr_prob)
    clicks_A = rng.binomial(n=group_A_views.astype(np.int64), p=group_A_CTR)
    clicks_B = rng.binomial(n=group_B.astype(np.int64), p=group_B_CTR) 
    t_stat, p_value = stats.ttest_ind(clicks_A, clicks_B, equal_var=False)
    p_values.append(p_value)

100%|██████████| 20000/20000 [04:39<00:00, 71.66it/s]


In [20]:
sum(np.array(p_values)<0.05) / 20000

0.2534

Результат: статистически значимый результат при таком наборе параметров из количества симуляций 20000, размере выборок (20998), длительности эксперимента (1 неделя), желаемом эффекте на просмотры тест будет показывать в 25,34% случаев. Мощность теста равна **0.2534**.

### Изменение № 1

Далее алгоритм был улучшен, теперь новый алгоритм охватывает больше пользователей. рекомендации отправляются всем пользователям с просмотрами больше 30. 

In [21]:
# улучшенный алгоритм, больше 30 просмотров
# эффект распространяется на большее число пользователей

p_values = []
for _ in tqdm(range(20000)): 
    group_A_views = rng.choice(views_distribution['views'], size= N, 
                           replace=True, p=views_distribution['pr'])
    group_B_views = rng.choice(views_distribution['views'], size=N, 
                           replace=True, p=views_distribution['pr'])
    group_B = group_B_views + ((1 + rng.binomial(n=1, p=0.5, size=N)) * rng.binomial(n=1, p=0.9, size=N) * (group_B_views >= 30))
    group_A_CTR=rng.choice(ctr, size=N, replace=True, p=ctr_prob)
    group_B_CTR=rng.choice(ctr, size=N, replace=True, p=ctr_prob)
    clicks_A = rng.binomial(n=group_A_views.astype(np.int64), p=group_A_CTR)
    clicks_B = rng.binomial(n=group_B.astype(np.int64), p=group_B_CTR) 
    t_stat, p_value = stats.ttest_ind(clicks_A, clicks_B, equal_var=False)
    p_values.append(p_value)

100%|██████████| 20000/20000 [04:38<00:00, 71.84it/s]


In [22]:
sum(np.array(p_values)<0.05) / 20000

0.41775

Результат: мощность теста повысилась. Он прокрашивается в **41,78%** случаев.

### Изменение № 2

Утвердили длительность эксперимента длиной в 2 недели. 
- Новые даты эксперимента: 22.05.2026 - 04.06.2026 (из задания)
- Пересчитываю размер групп с учетом приходящим новых пользователей
- Другие параметры аналогичны случаю после улучшения алгоритма (см. изменение 1)

In [23]:
# нужно поправить размер выборки, учитывая, что эксперимент 2 недели

q = """
SELECT views, count() as users
FROM (SELECT 
    user_id,
    sum(action = 'view') as views
FROM simulator_20260620.feed_actions 
WHERE toDate(time) between '2026-05-22' and '2026-06-04'
GROUP BY user_id
)
GROUP BY views
ORDER BY views
"""

N_new = ph.read_clickhouse(q, connection=connection).users.sum()

In [24]:
N_new = int(N_new / 2)

In [25]:
N_new 
# новый размер групп

30591

In [30]:
p_values = []
for _ in tqdm(range(20000)): 
    group_A_views = rng.choice(views_distribution['views'], size=N_new, 
                           replace=True, p=views_distribution['pr'])
    group_B_views = rng.choice(views_distribution['views'], size=N_new, 
                           replace=True, p=views_distribution['pr'])
    group_B = group_B_views + ((1 + rng.binomial(n=1, p=0.5, size=N_new)) * rng.binomial(n=1, p=0.9, size=N_new) * (group_B_views >= 30))
    group_A_CTR=rng.choice(ctr, size=N_new, replace=True, p=ctr_prob)
    group_B_CTR=rng.choice(ctr, size=N_new, replace=True, p=ctr_prob)
    clicks_A = rng.binomial(n=group_A_views.astype(np.int64), p=group_A_CTR)
    clicks_B = rng.binomial(n=group_B.astype(np.int64), p=group_B_CTR) 
    t_stat, p_value = stats.ttest_ind(clicks_A, clicks_B, equal_var=False)
    p_values.append(p_value)

100%|██████████| 20000/20000 [06:45<00:00, 49.36it/s]


In [31]:
sum(np.array(p_values)<0.05) / 20000

0.5624

Результат: при увеличении длительности эксперимента и размера выборок мощность теста повысилась до **0.5624**.

### Изменение № 3

Ранее я анализировала выборки целиком — и тех пользователей, на которых алгоритм повлиял, и тех, кого он не мог затронуть (меньше 30 просмотров). Сейчас буду отбирать только нужных пользователей и отправлять их в t-тест. Выборка будет меньше, но таким образом повысится чувствительность за счет исключения тех пользователей, которых алгоритм точно не затронет.

In [32]:
p_values = []
for _ in tqdm(range(20000)): 
    group_A_views = rng.choice(views_distribution['views'], size=N_new, 
                           replace=True, p=views_distribution['pr'])
    group_B_views = rng.choice(views_distribution['views'], size=N_new, 
                           replace=True, p=views_distribution['pr'])
    group_B = group_B_views + ((1 + rng.binomial(n=1, p=0.5, size=N_new)) * rng.binomial(n=1, p=0.9, size=N_new) * (group_B_views >= 30))
    group_A_CTR=rng.choice(ctr, size=N_new, replace=True, p=ctr_prob)
    group_B_CTR=rng.choice(ctr, size=N_new, replace=True, p=ctr_prob)
    clicks_A = rng.binomial(n=group_A_views.astype(np.int64), p=group_A_CTR)
    clicks_B = rng.binomial(n=group_B.astype(np.int64), p=group_B_CTR) 
    mask_A = group_A_views >= 30
    mask_B = group_B_views >= 30
    t_stat, p_value = stats.ttest_ind(clicks_A[mask_A], clicks_B[mask_B], equal_var=False)
    p_values.append(p_value)

100%|██████████| 20000/20000 [06:58<00:00, 47.80it/s]


In [33]:
sum(np.array(p_values)<0.05) / 20000

0.64835

Результат:  мощность теста повысилась до **0.6484**.

### Общий вывод

В рамках задания с опорой на симуляцию A/B эксперимента методом Монте-Карло я определила, как при изменении параметров проведения эксперимента меняется его способность выявлять различия между средним количеством лайков на пользователя в экспериментальной и контрольной группах по результатам введения в первой нового алгоритма рекомендаций.
1. Исходные параметры эксперимента: продолжительность 1 неделя, 20 тыс. симуляций, размере выборки 20998 (и участии в эксперименте выборки всех пользователей целиком), распространение алгоритма на пользователей с просмотрами больше 50, вероятность ошибки 5%. При исходных параметрах мощность составила **25.3%**.
2. Расширение числа пользователей, на которых распространился алгоритм, до тех, чьи просмотри больше 30, мощность возросла до **41.8%**.
3. Увеличение длительности эксперимента до 2 недель и рост размера выборки (30591) мощность повысилась до **56.2%**.
4. Фильтрация по нужным пользователям, которых точно мог затронуть алгоритм, до непосредственного проведения t-теста, повысила чувствительность теста и его мощность до **64.8%**, несмотря на сокращение размера выборок.

На текущем примере видно, что с ростом продолжительности эксперимента, размера выборки, чувствительности теста растет процент случаев, когда он может зафиксировать статистически значимые различия между средними значениями по группам. При этом, например, повышение чувствительности может происходить при сокращении размера выборки, но приводит к росту мощности (как в рассматриваемом случае). Параметры теста компенсируют друг друга: повышение чувствительности теста требует меньшего размера выборки.

По итогам самого последнего улучшения мы все еще может выявить статистически значимые различия только в 64.8% (когда стандарт мощности 80%). Решение о том, чтобы выкатывать изменение на всех пользователей, принимать рано. Вероятно, для повышения мощности необходимо увеличить количество симуляций или поработать над новым алгоритмом так, чтобы он охватывал пользователей и с меньшим количеством просмотров.